# Download and preprocess all public daily snow station data into Zarr format

Data source: [egagli/global_snow_networks](https://github.com/egagli/global_snow_networks) — a daily-refreshed archive of daily snow point observations from multiple public networks (SNOTEL, SNOTEL Lite, CCSS/CDEC, BC Snow Survey, NVE Norway, Yukon, SCAN, COOP, and AWDB's "MSNT" bucket).

Two files, both fetched anonymously from the public repo:
- [`all_snow_stations.geojson`](https://github.com/egagli/global_snow_networks/blob/main/all_snow_stations.geojson) — the **combined** station inventory (metadata): every known station from every client, including periodic snow courses and other manual sites. Stations with a probe-verified daily-or-better record carry `daily_or_better: true`, and the CSV archive covers essentially exactly those.
- [`data/all_station_csvs.tar.xz`](https://github.com/egagli/global_snow_networks/blob/main/data/all_station_csvs.tar.xz) — one CSV per daily station (`date, wteq_cm, snwd_cm`)

This notebook downloads both, extracts the CSVs, combines everything into a single `(time, station_id)` xarray Dataset, and writes it to Zarr. The **CSV archive defines the station set**; the inventory is joined on for metadata.

**Units:** SWE is converted from cm to **mm** (kg m⁻²), snow depth stays in cm.

In [ ]:
import requests
import tarfile
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import easysnowdata
import matplotlib.pyplot as plt
import contextily as ctx

data_dir = Path("data/snow_pillows")
data_dir.mkdir(parents=True, exist_ok=True)

geojson_filename = "all_snow_stations.geojson"
geojson_filepath = data_dir / geojson_filename

tar_filename = "all_station_csvs.tar.xz"
tar_filepath = data_dir / tar_filename
csv_dir = data_dir / "stations"

zarr_filepath = data_dir / "snow_pillows.zarr"

base_url = "https://raw.githubusercontent.com/egagli/global_snow_networks/main"

In [ ]:
# Get station inventory + CSV bundle (~65 MB total, skipped if already present).
# The source repo is public, so these are plain anonymous raw.githubusercontent.com
# requests -- no token, no local clone needed.
for remote_path, filepath in [(geojson_filename, geojson_filepath),
                              (f"data/{tar_filename}", tar_filepath)]:
    if filepath.exists():
        print(f"{filepath} already exists")
        continue

    print(f"Downloading {remote_path}...")
    with requests.get(f"{base_url}/{remote_path}", stream=True) as r:
        r.raise_for_status()
        with open(filepath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    print(f"Downloaded -> {filepath} ({filepath.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Extract CSVs (tarball contains stations/*.csv)
if not csv_dir.exists() or not any(csv_dir.glob("*.csv")):
    print("Extracting...")
    with tarfile.open(tar_filepath, 'r:xz') as tf:
        tf.extractall(data_dir)
    print("Extraction complete")
else:
    print("Already extracted")

csv_files = sorted(csv_dir.glob("*.csv"))
print(f"{len(csv_files)} station CSVs")

In [ ]:
# Read only the fields needed -- the subset also skips the nested JSON columns
# (data_variables, possible_duplicates, drainage_basin_key) that pyogrio can't parse.
inventory_cols = ['code', 'name', 'network_code', 'client', 'state',
                  'latitude', 'longitude', 'elevation_m', 'daily_or_better']
stations_gdf = gpd.read_file(geojson_filepath, columns=inventory_cols)

csv_codes = {f.stem for f in csv_files}
print(f"{len(stations_gdf)} stations in the combined inventory, "
      f"{stations_gdf['daily_or_better'].sum()} flagged daily_or_better, "
      f"{stations_gdf['code'].isin(csv_codes).sum()} matched to a CSV in the archive")

stations_gdf.loc[stations_gdf['code'].isin(csv_codes), 'network_code'].value_counts()

## Combine all station CSVs into a single (time, station_id) Dataset

Read every CSV, then fill two preallocated `(time, station_id)` float32 arrays by positional date index (much faster than pandas alignment across 1500+ misaligned series).

Dates are parsed with `errors='coerce'` and NaT rows dropped as a guard: an earlier snapshot of the archive had a few hundred truncated date strings from a refresh artifact. That has since been fixed upstream, so this should now report zero dropped rows.

In [ ]:
%%time
frames = {}
n_bad_dates = 0
for f in csv_files:
    df = pd.read_csv(f)
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')
    n_bad_dates += int(df['date'].isna().sum())
    df = df.dropna(subset=['date']).drop_duplicates(subset='date', keep='first').set_index('date')
    frames[f.stem] = df

print(f"Read {len(frames)} CSVs, dropped {n_bad_dates} rows with corrupted dates")

In [ ]:
%%time
# The CSV archive defines the station set. Inventory metadata and the CSV archive are
# refreshed by separate pipeline stages, so a brand-new CSV can briefly lack an
# inventory entry -- drop those rather than hard-failing on the join.
inventory = stations_gdf.set_index('code')
orphans = sorted(set(frames) - set(inventory.index))
if orphans:
    print(f"Dropping {len(orphans)} CSVs with no inventory entry: {orphans}")
    frames = {code: df for code, df in frames.items() if code not in orphans}

tmin = min(df.index.min() for df in frames.values())
tmax = max(df.index.max() for df in frames.values())
full_index = pd.date_range(tmin, tmax, freq='D')
print(f"time range {tmin.date()} .. {tmax.date()} ({len(full_index)} days)")

station_ids = np.array(list(frames.keys()))
swe = np.full((len(full_index), len(station_ids)), np.nan, dtype='float32')
snow_depth = np.full_like(swe, np.nan)
for j, sid in enumerate(station_ids):
    df = frames[sid]
    pos = (df.index - tmin).days.values
    swe[pos, j] = df['wteq_cm'].values * 10.0   # cm -> mm (kg m-2), matches NorSWE 'snw'
    snow_depth[pos, j] = df['snwd_cm'].values   # keep cm

meta = inventory.reindex(station_ids)
assert meta['name'].notna().all(), "CSV without matching inventory entry"

# .to_numpy(dtype=...) rather than .values: pandas 3 string columns are Arrow-backed,
# and .values/.astype(str) return an ArrowStringArray whose StringDtype dask can't chunk
snow_pillows_ds = xr.Dataset(
    {
        'swe':        (('time', 'station_id'), swe),
        'snow_depth': (('time', 'station_id'), snow_depth),
    },
    coords={
        'time':         full_index,
        'station_id':   station_ids.astype(str),
        'station_name': ('station_id', meta['name'].to_numpy(dtype=str)),
        'network':      ('station_id', meta['network_code'].to_numpy(dtype=str)),
        'client':       ('station_id', meta['client'].to_numpy(dtype=str)),
        'state':        ('station_id', meta['state'].fillna('').to_numpy(dtype=str)),
        'latitude':     ('station_id', meta['latitude'].to_numpy('float64')),
        'longitude':    ('station_id', meta['longitude'].to_numpy('float64')),
        'elevation':    ('station_id', meta['elevation_m'].to_numpy('float32')),
    },
)
snow_pillows_ds['swe'].attrs = {'units': 'mm', 'long_name': 'snow water equivalent'}
snow_pillows_ds['snow_depth'].attrs = {'units': 'cm', 'long_name': 'snow depth'}
snow_pillows_ds['elevation'].attrs = {'units': 'm'}
snow_pillows_ds.attrs['source_url'] = 'https://github.com/egagli/global_snow_networks'
snow_pillows_ds.attrs['description'] = ('Daily SWE and snow depth from public snow networks '
                                        '(SNOTEL, SNOTEL Lite, CCSS/CDEC, BC Snow Survey, NVE, '
                                        'Yukon, SCAN, COOP, MSNT)')
snow_pillows_ds

In [ ]:
# how much memory does the dataset take up in RAM when loaded?
print(f"Estimated RAM usage when loaded: {snow_pillows_ds.nbytes / 1e9:.2f} GB")

In [ ]:
if zarr_filepath.exists():
    shutil.rmtree(zarr_filepath)
    print(f'{zarr_filepath} already exists, deleted for fresh conversion')

snow_pillows_ds.chunk({'time': 4000, 'station_id': -1}).to_zarr(zarr_filepath, mode='w')

zarr_size_mb = sum(f.stat().st_size for f in zarr_filepath.rglob("*") if f.is_file()) / 1e6
print(f"Zarr size on disk: {zarr_size_mb:.1f} MB")

## Confirm Zarr version of the dataset is complete and remove the tarball + extracted CSVs

In [ ]:
snow_pillows_ds = xr.open_zarr(zarr_filepath)
snow_pillows_ds

In [ ]:
if csv_dir.exists():
    shutil.rmtree(csv_dir)
    print(f'{csv_dir} deleted to save space')
if tar_filepath.exists():
    tar_filepath.unlink()
    print(f'{tar_filepath} deleted to save space')

## Inspect distribution of sites

In [ ]:
snow_pillows_gdf = gpd.GeoDataFrame(
    {
        'station_id':   snow_pillows_ds['station_id'].values,
        'station_name': snow_pillows_ds['station_name'].values,
        'network':      snow_pillows_ds['network'].values,
        'client':       snow_pillows_ds['client'].values,
        'state':        snow_pillows_ds['state'].values,
        'elevation':    snow_pillows_ds['elevation'].values,
    },
    geometry=gpd.points_from_xy(snow_pillows_ds['longitude'].values, snow_pillows_ds['latitude'].values),
    crs='EPSG:4326',
)

snow_pillows_gdf

In [ ]:
snow_pillows_gdf.explore(
    column='network',
    tooltip=['station_id', 'station_name', 'network', 'client', 'state', 'elevation'],
    marker_kwds={'radius': 4},
)

## Sanity check: compare against the easysnowdata SNOTEL archive for the same site (Paradise, WA)

In [ ]:
StationsWUS = easysnowdata.automatic_weather_stations.StationCollection()
StationsWUS.get_entire_data_archive()
paradise_easysnowdata_swe_mm_da = 1000.0 * StationsWUS.entire_data_archive['WTEQ'].sel(station="679_WA_SNTL")  # m -> mm
paradise_pillow_swe_da = snow_pillows_ds['swe'].sel(station_id='679_WA_SNTL')

In [ ]:
f, ax = plt.subplots(figsize=(15, 4))
paradise_pillow_swe_da.plot(ax=ax, label='global_snow_networks', color='blue', zorder=1, linewidth=2)
paradise_easysnowdata_swe_mm_da.plot(ax=ax, label='easysnowdata archive', color='red', zorder=2, linewidth=1)
ax.set_title('Paradise SNOTEL (679_WA_SNTL) SWE [mm]')
ax.legend()

In [ ]:
f, ax = plt.subplots(figsize=(15, 4))
paradise_pillow_swe_da.plot(ax=ax, label='global_snow_networks', color='blue', zorder=1, linewidth=2)
paradise_easysnowdata_swe_mm_da.plot(ax=ax, label='easysnowdata archive', color='red', zorder=2, linewidth=1)
ax.set_xlim([pd.to_datetime('2020-01-01'), pd.to_datetime('2022-12-31')])
ax.set_title('Paradise SNOTEL (679_WA_SNTL) SWE [mm]')
ax.legend()